In [84]:
!pip install bert-score transformers torch sentence-transformers numpy spacy

## Imports

In [85]:
import os
import ast
import nltk
import json
import torch
import spacy
import ollama
import numpy as np
import torch.nn.functional as F
from rapidfuzz import fuzz
from bert_score import score
from transformers import logging, pipeline

from sentence_transformers import SentenceTransformer, util
from transformers import BartTokenizer, BartForConditionalGeneration, AutoTokenizer, AutoModelForSequenceClassification


In [86]:
logging.set_verbosity_error()

## Contextual Semantic Metrics 

In [87]:
def calculate_bertscore(paragraph:str, rewrite:str, type="bert"):
    if type == "bert":
        model = "bert-base-uncased"
    if type == "scibert":
        model = "allenai/scibert_scivocab_uncased"

    P, R, F1 = score(
    cands= rewrite,
    refs= paragraph,
    lang="en",
    model_type=model,
    verbose=False,
    )
    
    return F1.mean().item()

In [88]:
def calculate_bartscore(paragraph:str, rewrite:str):
    model_name = "facebook/bart-large"
    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name)
    model.eval()

    with torch.no_grad():
        inputs = tokenizer(paragraph, return_tensors="pt", truncation=True)
        labels = tokenizer(rewrite, return_tensors="pt", truncation=True)["input_ids"]

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss  # cross-entropy
        
        bart_score = -loss.item()

    return bart_score

In [89]:
def calculate_cosine_similarity(paragraph:str, rewrite:str):
    model_name = "sentence-transformers/all-mpnet-base-v2"
    model = SentenceTransformer(model_name)
    
    embeddings = model.encode([paragraph, rewrite], show_progress_bar=False)
    
    vec1 = embeddings[0]
    vec2 = embeddings[1]
    
    cosine = np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )
    return float(cosine)

In [90]:
def calculate_semantic_metrics(paragraph, rewrite):
    bert_score = calculate_bertscore(paragraph, rewrite)
    #scibert_score = calculate_bertscore(paragraph, rewrite, "scibert")
    bart_score = calculate_bartscore(paragraph, rewrite)
    cosinus_similarity = calculate_cosine_similarity(paragraph[0], rewrite[0])
    return {"BertScore":bert_score,
            #"SciBertScore": scibert_score,
            "BartScore": bart_score,
            "Cos_Sim":cosinus_similarity}

## Factual metrics

In [91]:
tokenizer = AutoTokenizer.from_pretrained("manueldeprada/FactCC")
model = AutoModelForSequenceClassification.from_pretrained("manueldeprada/FactCC")

def factcc_score(paragraph, rewrite):

    inputs = tokenizer(
        paragraph,
        rewrite,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1)

    return prediction.item()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 497.73it/s, Materializing param=classifier.weight]                                      


In [92]:
#spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

In [93]:
def factacc_score(paragraph, rewrite):

    doc = nlp(rewrite)

    facts = []
    correct = 0

    for token in doc:
        if token.dep_ == "nsubj":
            subject = token.text
            verb = token.head.text

            for child in token.head.children:
                if child.dep_ == "dobj":
                    obj = child.text
                    facts.append((subject, verb, obj))

    for fact in facts:
        if all(word.lower() in paragraph.lower() for word in fact):
            correct += 1

    if len(facts) == 0:
        return 0

    return correct / len(facts)

In [94]:
def dae_score(paragraph, rewrite):

    doc = nlp(rewrite)

    arcs_total = 0
    arcs_supported = 0

    for token in doc:
        arcs_total += 1

        relation = f"{token.head.text} {token.dep_} {token.text}"

        if relation.lower() in paragraph.lower():
            arcs_supported += 1

    if arcs_total == 0:
        return 0

    return arcs_supported / arcs_total

In [ ]:
# -----------------------------
# Fonction LLM
# -----------------------------
def llm(prompt):

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {"role": "user", "content": prompt}
        ],
         options = {
                "temperature": 0.9,  # sortie stable
                     }
                )
    
    return response["message"]["content"]


In [96]:
# -----------------------------
# Extraction des faits
# -----------------------------

def extract_facts(text):

    prompt = f"""
    Extract atomic factual statements from the following text.

    Text:
    {text}

    Instructions:
    - Return ONLY a valid Python list of strings.
    - Do NOT include any code, function, variable name, or explanation.
    - Do NOT wrap the result in markdown.
    - Do NOT add any text before or after the list.
    - Each element must be a short factual sentence.
    - If no facts are found, return [].

    Output example:
    ["Fact 1", "Fact 2"]

    Now return the result:
    """

    response = llm(prompt) 

    facts = response 

    return facts

In [ ]:
def naive_fact_score(source_facts, summary_facts):
    if not summary_facts:
        print("\nHere")
        return 0.0

    supported = 0
    for fact in summary_facts:
        if any(fuzz.partial_ratio(fact.lower(), s.lower()) > 85 for s in source_facts):
            supported += 1
    
    return supported / len(summary_facts)

# modèle NLI
nli = pipeline("text-classification", model="facebook/bart-large-mnli")

def classify_nli(premise, hypothesis):
    """
    premise = source fact
    hypothesis = summary fact
    """
    result = nli(f"{premise} </s> {hypothesis}")[0]
    return result["label"], result["score"]

def nli_fact_based_score(source_facts, summary_facts):
    supported = 0
    contradicted = 0
    neutral = 0

    for h in summary_facts:
        best_label = "neutral"
        best_score = 0

        for p in source_facts:
            label, score = classify_nli(p, h)

            if score > best_score:
                best_label = label
                best_score = score

        if best_score <= 0.5:
            neutral += 1
        elif best_label == "ENTAILMENT":
            supported += 1
        elif best_label == "CONTRADICTION":
            contradicted += 1
        else:
            neutral += 1

    total = len(summary_facts)

    if total == 0:
        return 0.0

    return max(0,(supported - contradicted) / total)

def cos_sim_fact_based_score(source_facts, summary_facts):

    model = SentenceTransformer('all-MiniLM-L6-v2')

    supported = 0

    for fact in summary_facts:
        fact_emb = model.encode(fact, convert_to_tensor=True)

        for s in source_facts:
            s_emb = model.encode(s, convert_to_tensor=True)
            sim = util.cos_sim(fact_emb, s_emb)

            if sim > 0.75:
                supported += 1
                break

    return supported / len(summary_facts) if summary_facts else 0
   

Loading weights: 100%|██████████| 515/515 [00:01<00:00, 415.63it/s, Materializing param=model.shared.weight]                                   


In [ ]:
# -----------------------------
# FactScore
# -----------------------------
def factscore(source, summary):

    source_facts = ast.literal_eval(extract_facts(source))
    summary_facts = ast.literal_eval(extract_facts(summary))

    print(f"SUMMARY FACTS \n\n {summary_facts}")
    print(f"SOURCE FACTS \n\n {source_facts}")

    naive_score = naive_fact_score(source_facts, summary_facts)
    cos_sim_score = cos_sim_fact_based_score(source_facts, summary_facts)
    nli_score = nli_fact_based_score(source_facts, summary_facts)

    final_score = 0.25*naive_score + 0.25*cos_sim_score + 0.5*nli_score

    print(f"Naive Score {naive_score} | Cosim Score : {cos_sim_score} | NLI Score: {nli_score}")
    return final_score

In [104]:
def calculate_factual_metrics(paragraph, rewrite):
    fact_cc = factcc_score(paragraph, rewrite)
    fact_acc = factacc_score(paragraph, rewrite)
    dae = dae_score(paragraph, rewrite)
    fact_score = factscore(paragraph, rewrite)

    return {"FactCC":fact_cc,
            "FactAcc": fact_acc,
            "DAE":fact_score,
            "FActScore":fact_score}

In [105]:
def keys_update(dico,index=1):
    new_dico = {f"{k}{index}": v for k, v in dico.items()}
    return new_dico

In [106]:
def compute_all_metrics(data):
    for article in data:
        for i in range(1,4):

            semantic_metrics = calculate_semantic_metrics([article['Paragraphes']], [article[f"Rew{i}"]])
            semantic_metrics_updated = keys_update(semantic_metrics, i)
            article.update(semantic_metrics_updated)

            factual_metrics = calculate_factual_metrics(article['Paragraphes'], article[f"Rew{i}"])
            factual_metrics_updated = keys_update(factual_metrics,i)
            article.update(factual_metrics_updated)
    return data

## Code Tests

In [107]:
with open("../data/json/rewrites.json","r" , encoding='utf-8') as file:
    data = json.load(file)

In [108]:
filename = "../data/json/metrics.json"
if not os.path.exists(filename):
    with open(filename, "w", encoding="utf-8") as f:
        pass

In [109]:
all_metrics = compute_all_metrics(data)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 379.50it/s, Materializing param=pooler.dense.weight]                        


SUMMARY FACTS 

 ['A Hidden Markov Model is a classical statistical model.', 'It is often used in NLP for tagging and can be applied to speech recognition, part-of-speech tagging, and various bioinformatics applications.', 'Given an input sequence of n signs X = (x1, …, xn) and a possible output transliteration Y = (y1, …, yn), an HMM provides the probability p(X, Y).', 'Thus, given a sequence of signs X = (x1, …, xn), we can choose the transliteration that maximizes the probability.']
SOURCE FACTS 

 ['A Hidden Markov Model (HMM) is a classical statistical model.', 'HMM is often used in NLP for tagging.', 'A HMM can be applied to speech recognition, part-of-speech tagging and various bioinformatics applications.', 'Given an input sequence of n signs X = (x1, …, xn), a HMM provides the probability p(X, Y).']


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 396.84it/s, Materializing param=pooler.dense.weight]                             


Naive Score 0.5 | Cosim Score : 0.5 | NLI Score: 0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 371.77it/s, Materializing param=pooler.dense.weight]                        


...

Naive Score 1.0 | Cosim Score : 1.0 | NLI Score: 0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 341.17it/s, Materializing param=pooler.dense.weight]                        


SUMMARY FACTS 

 ['A Hidden Markov Model is a classical statistical model.', 'It can be applied to speech recognition, part-of-speech tagging and various bioinformatics applications.', 'Given an input sequence of n signs X = (x1, …, xn), an HMM provides the probability p(X, Y).']
SOURCE FACTS 

 ['A Hidden Markov Model (HMM) is a classical statistical model.', 'A HMM embodies a set of statistical assumptions concerning the generation of sample data and similar data from a larger population.', 'It can be applied to speech recognition, part-of-speech tagging and various bioinformatics applications inter alia.', 'An HMM provides the probability p(X, Y) for an input sequence of n signs X = (x1, …, xn) and a possible output transliteration Y = (y1, …, yn).', 'A given sequence of signs X = (x1, …, xn) can choose the transliteration that maximizes the probability.']


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 362.65it/s, Materializing param=pooler.dense.weight]                             


Naive Score 0.6666666666666666 | Cosim Score : 1.0 | NLI Score: 0


In [112]:
all_metrics

[{'Doi': 'https://doi.org/10.57896/2024-tal-65_3_3',
  'Paragraphes': 'A Hidden Markov Model (HMM) is a classical statistical model [32] that embodies a set of statistical assumptions concerning the generation of sample data (and similar data from a larger population). It is often used in NLP for tagging [33, 34]. It can be applied to speech recognition, part-of-speech tagging and various bioinformatics applications inter alia. Given an input sequence of n signs X = (x1, …, xn) and a possible output transliteration Y = (y1, …, yn), an HMM provides the probability p(X, Y). Thus, given a sequence of signs X = (x1, …, xn), we can choose the transliteration  that maximizes .',
  'Auteur': 'Delphine Battistelli, Farah Benamara, Viviana Patti',
  'Date de publication': '2025',
  'URL': 'https://aclanthology.org/2024.tal-3.4/',
  'Licence': 'CC BY 4.0',
  'Domaine': 'TALN',
  'Rew1': 'A Hidden Markov Model (HMM) is a classical statistical model that embodies a set of statistical assumptions c

In [114]:
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=4, ensure_ascii=False)